# 00 · Start here

Run these cells to see that Spark can talk to Kafka. Start the replayer first, in a terminal:

```
python producers/replay.py --from 2026-10-01 --to 2026-10-01 --speed 0
```

This notebook only *looks* at the data. Building the pipeline is up to you (Q3 onward).

In [ ]:
from sbetl.common.spark import get_spark
from sbetl.common.config import load_config

cfg = load_config()
spark = get_spark('explore')
spark

In [ ]:
# A batch read of a topic: what does a Kafka record look like in Spark?
raw = (spark.read.format('kafka')
       .option('kafka.bootstrap.servers', cfg['kafka']['bootstrap'])
       .option('subscribe', cfg['kafka']['topics']['player'])
       .option('startingOffsets', 'earliest')
       .option('includeHeaders', 'true')
       .load())
raw.printSchema()

In [ ]:
raw.selectExpr('partition', 'offset', 'timestamp', 'CAST(key AS STRING) AS key', 'CAST(value AS STRING) AS value', 'headers').show(5, truncate=80)

In [ ]:
# How are messages spread across partitions? Why is the key the user ID?
raw.groupBy('partition').count().orderBy('partition').show()

Questions to think about before Q3:

1. The value is bytes. Where does the schema live?
2. `timestamp` is the Kafka time. Is that when the viewing event happened?
3. What happens to this notebook's numbers if you run the replayer twice?